# Game Simulator
Forecast DAU, payer DAU, and revenue over a 365-day horizon by adjusting UA spend, CPI, retention, and conversion inputs.

In [1]:
# show
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import date

from common_lib.sql import BigQueryConnector
from common_lib.sheets import load_inputs, get_inputs_dir
from common_lib.simulation import (
    SimulationEngine, PlatformInputs,
    save_scenario, load_scenario, list_scenarios,
    save_result, load_result, list_results,
)
from common_lib.widgets import ScenarioPanel
from common_lib.tables import monthly_table

#print('Inputs dir:', get_inputs_dir())

In [3]:
refresh_data = True  # Set to True to refresh data from BigQuery, False to load from local pickle

In [4]:
# show
actuals_params = {'start_date': '2021-06-01'}
if refresh_data:
    bqc = BigQueryConnector()
    bqc.print_cost_estimate('./sql/actuals.sql', is_path=True, query_parameters=actuals_params)
else:
    bqc = None

This query will process 16.88 GB when run.
Estimated query cost: $0.11


In [4]:
PLATFORM_MAP = {'AND': 'android', 'IOS': 'ios'}

if refresh_data == True:
    actuals = bqc.get('./sql/actuals.sql', is_path=True, query_parameters=actuals_params)
    pd.to_pickle(actuals, './data/actuals.pkl')
else:
    actuals = pd.read_pickle('./data/actuals.pkl')

actuals['dt'] = pd.to_datetime(actuals['dt'])
actuals['platform'] = actuals['platform'].map(PLATFORM_MAP).fillna(actuals['platform'].str.lower())
actuals = actuals.sort_values('dt')

# Anchor DAU: last observed day per platform
anchor_dau = actuals.sort_values('dt').groupby('platform')['dau'].last().to_dict()

In [5]:
# show
cohort_params = {'start_date': '2021-06-01'}
if refresh_data:
    bqc.print_cost_estimate('./sql/retention.sql',  is_path=True, query_parameters=cohort_params)
    bqc.print_cost_estimate('./sql/conversion.sql', is_path=True, query_parameters=cohort_params)

This query will process 10.10 GB when run.
Estimated query cost: $0.07
This query will process 10.12 GB when run.
Estimated query cost: $0.07


In [6]:


if refresh_data == True:
    live_retention  = bqc.get('./sql/retention.sql',  is_path=True, query_parameters=cohort_params)
    live_retention.to_pickle('./data/live_retention.pkl')
    live_conversion = bqc.get('./sql/conversion.sql', is_path=True, query_parameters=cohort_params)
    live_conversion.to_pickle('./data/live_conversion.pkl')
else:
    live_retention  = pd.read_pickle('./data/live_retention.pkl')
    live_conversion = pd.read_pickle('./data/live_conversion.pkl')

live_retention['platform']  = live_retention['platform'].map(PLATFORM_MAP).fillna(live_retention['platform'].str.lower())
live_conversion['platform'] = live_conversion['platform'].map(PLATFORM_MAP).fillna(live_conversion['platform'].str.lower())

In [ ]:
# show
if refresh_data:
    bqc.print_cost_estimate('./sql/installs.sql', is_path=True)

In [ ]:
if refresh_data:
    installs = bqc.get('./sql/installs.sql', is_path=True)
    installs['platform'] = installs['platform'].map(PLATFORM_MAP).fillna(installs['platform'].str.lower())
    installs['dt'] = pd.to_datetime(installs['dt'])
    pd.to_pickle(installs, './data/installs.pkl')
else:
    installs = pd.read_pickle('./data/installs.pkl')

In [7]:
# show
sheet_inputs = load_inputs()

for name, df in sheet_inputs.items():
    print(f'\n--- {name} ---')
    #print(df.to_string(index=False))


--- cpi ---

--- ua_spend ---

--- team_cost ---


## 4. Interactive Scenario Panel

In [ ]:
# show
from common_lib.app import prefill_panel, setup_callbacks

engine = SimulationEngine()
panel  = ScenarioPanel(saved_scenarios=list_scenarios())
prefill_panel(panel, actuals, anchor_dau, sheet_inputs)
setup_callbacks(panel, engine, actuals,
                live_retention=live_retention, live_conversion=live_conversion,
                installs=installs,
                default_scenario='forecast_file_20260512')
panel.display()

In [9]:
from common_lib.plots import plot, plot_retention, plot_conversion, configure as configure_plots
from common_lib.simulation import list_results

configure_plots(actuals)

# ── Usage ──────────────────────────────────────────────────────────────────
# plot('base_case')                         # all charts
# plot('base_case', chart='dau')            # DAU only
# plot('base_case', chart='revenue')        # daily revenue
# plot('base_case', chart='monthly')        # monthly bar
# plot(['base_case', 'high_ua'])            # compare scenarios
#
# plot_retention('base_case')               # retention curve from saved scenario
# plot_conversion('base_case')              # conversion curve from saved scenario
# plot_retention(['base_case', 'high_ua'])  # compare retention curves across scenarios
# plot_retention(panel.get_curve_anchors()) # preview current panel state
#print('Available results:', list_results())

In [10]:
#plot('test3', chart='dau')

In [11]:
#plot('test3', chart='revenue')

In [12]:
#plot_conversion('test3')
#plot_retention('test3') 

In [13]:
# show
def summary_table(scenarios=None) -> pd.DataFrame:
    """
    Summarise saved simulation results.
    scenarios: list of names, or None to include all saved results.
    """
    names = scenarios if scenarios is not None else list_results()
    rows = []
    for name in names:
        df = load_result(name)
        for platform in ('ios', 'android', 'combined'):
            sub = df[df['platform'] == platform]
            if sub.empty:
                continue
            rows.append({
                'scenario':       name,
                'platform':       platform,
                'avg_dau':        round(sub['dau'].mean()),
                'peak_dau':       round(sub['dau'].max()),
                'total_installs': round(sub['new_installs'].sum()),
                'total_iap_rev':  round(sub['iap_revenue'].sum(), 2),
                'total_ad_rev':   round(sub['ad_revenue'].sum(), 2),
                'total_revenue':  round(sub['total_revenue'].sum(), 2),
            })
    return pd.DataFrame(rows)


#summary_table()